# Lab 5 — Water Quality and Algae Detection: Airborne–Satellite Fusion

This notebook demonstrates a complete workflow for monitoring water quality and
detecting algal blooms by fusing airborne hyperspectral data with Sentinel-2
multispectral imagery.

**Objective.** Derive water-quality and algae detection proxy indices from both
an airborne ENVI scene and a matching Sentinel-2 acquisition, compare the two
data sources on a common grid, and apply Spectral Angle Mapping (SAM) for
land-cover classification after cross-sensor calibration.

**Pipeline overview:**

1. Load and inspect the airborne hyperspectral scene.
2. Build natural RGB and false-color composites.
3. Compute 9 airborne index maps (water-quality + algae detection).
4. Apply the NDWI water mask and analyse algae indices over water pixels.
5. Search for and load the closest Sentinel-2 scene.
6. Align both data sources to a shared grid and compare 7 cross-sensor indices.
7. Load the spectral library built with the interactive viewer.
8. Calibrate Sentinel-2 reflectance against airborne bands.
9. Run SAM classification on both sensors and compare class assignments.

All processing logic lives in reusable modules under `src/lab5/`. The notebook
orchestrates these modules and produces the final visualisations and tables.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

if (Path.cwd() / "src").exists():
    REPO_ROOT = Path.cwd()
else:
    REPO_ROOT = Path.cwd().parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.lab5.envi_utils import (
    build_invalid_band_mask,
    find_nearest_band,
    get_ignore_value,
    get_rgb_bands,
    load_envi_image,
    parse_wavelengths,
    read_rgb,
)
from src.lab5.indices import (
    band_by_wavelength,
    chlorophyll_red_edge_peak,
    composite_by_wavelengths,
    doc_proxy_green_red,
    fai,
    flh,
    mci,
    ndci,
    ndre,
    ndti,
    ndwi,
    stretch_composite,
)
from src.lab5.sentinel2 import (
    SENTINEL2_REQUIRED_BANDS,
    align_array_to_grid,
    build_overlap_mask,
    comparison_output_dir,
    describe_item,
    download_band,
    find_cached_band_files,
    load_local_stack,
    open_catalog,
    scene_bbox_wgs84_from_envi,
    scene_grid_from_envi,
    search_s2_items,
    select_best_item,
    summarize_pair,
)
from src.lab5.sam import (
    SENTINEL2_SAM_BANDS,
    align_reference_to_wavelengths,
    apply_bandwise_linear_calibration,
    fit_bandwise_linear_calibration,
    load_library_summaries,
    resample_airborne_spectrum_to_s2,
    sam_classification,
    spectral_angle_image,
    stack_named_bands,
)

SCENE_PATH = REPO_ROOT / "data/images/221000_Odra_HS_Blok_A_008_VS_join_atm.hdr"
SPECTRAL_LIBRARY_DIR = REPO_ROOT / "data/spectral_library"
SENTINEL2_DIR = REPO_ROOT / "data/sentinel2"
OUTPUTS_ROOT = REPO_ROOT / "data/outputs"
TARGET_DATE = "2025-06-17"
S2_DAY_WINDOW = 5
S2_CLOUD_LIMIT = 20.0

SCENE_PATH

## 1. Scene Summary

The first step loads the airborne hyperspectral ENVI scene, reads the wavelength
vector from the header, identifies invalid (water-absorption / sensor-gap) bands,
and resolves the nearest usable airborne band for each target wavelength used by
the Lab 5 products.

The target wavelengths correspond to the Sentinel-2 bands used for cross-sensor
comparison plus the narrow-band airborne wavelengths needed for the
fluorescence-based algae indices:

| Target nm | Sentinel-2 band | Use |
|-----------|-----------------|-----|
| 490       | B02 (blue)      | Water composite |
| 560       | B03 (green)     | DOC proxy, NDTI, NDWI |
| 665       | B04 (red)       | Chl-a, NDCI, NDTI, FAI |
| 681       | -- (airborne only) | FLH, MCI |
| 705       | B05 (red edge)  | Chl-a red-edge peak, NDCI, NDRE |
| 709       | -- (airborne only) | FLH |
| 740       | B06 (red edge)  | Chl-a red-edge peak |
| 753       | -- (airborne only) | MCI |
| 842       | B08 (NIR)       | NDRE, FAI, NDWI, vegetation composite |
| 1610      | B11 (SWIR)      | FAI, moisture composite |

In [ ]:
img = load_envi_image(SCENE_PATH)
meta = img.metadata
wavelengths = parse_wavelengths(meta)
ignore_value = get_ignore_value(meta)
invalid_mask = build_invalid_band_mask(meta)
rgb_default = read_rgb(img, get_rgb_bands(meta), ignore_value)

def resolve_target(target_nm: float) -> tuple[int, float]:
    band_index = find_nearest_band(wavelengths, target_nm, invalid_mask)
    return band_index, float(wavelengths[band_index])

target_lookup = {
    target_nm: resolve_target(target_nm)
    for target_nm in (490, 560, 665, 681, 705, 709, 740, 753, 842, 1610)
}

metadata_summary = pd.DataFrame([
    {
        "scene_id": SCENE_PATH.stem,
        "rows": img.nrows,
        "cols": img.ncols,
        "bands": img.nbands,
        "ignore_value": ignore_value,
        "invalid_band_count": int(invalid_mask.sum()) if invalid_mask is not None else 0,
        "wavelength_count": int(len(wavelengths)) if wavelengths is not None else 0,
    }
])

band_summary = pd.DataFrame(
    [
        {
            "target_nm": target_nm,
            "band_index": band_index,
            "actual_wavelength_nm": actual_wavelength,
        }
        for target_nm, (band_index, actual_wavelength) in sorted(target_lookup.items())
    ]
)

display(metadata_summary)
display(band_summary)

## 2. Natural and False-Color Composites

False-color composites highlight features that are not visible in natural RGB.
The natural RGB uses the ENVI `default bands` metadata. The three false-color
composites follow the Lab 5 execution plan:

- **Vegetation composite (842 / 665 / 560):** Healthy vegetation appears bright
  red because it strongly reflects NIR (842 nm) while absorbing red (665 nm).
  Water and bare soil appear dark.
- **Moisture composite (1610 / 842 / 665):** SWIR reflectance (1610 nm) is
  sensitive to moisture content. Wet areas appear dark; dry vegetation and soil
  are brighter. Useful for separating water from land.
- **Water-focused composite (705 / 560 / 490):** The red-edge (705 nm) and blue
  (490 nm) channels emphasise chlorophyll fluorescence and scattering in water
  bodies, making turbidity and algal blooms more visible.

In [ ]:
vegetation_rgb = stretch_composite(
    composite_by_wavelengths(
        img,
        wavelengths,
        (842, 665, 560),
        invalid_mask=invalid_mask,
        ignore_value=ignore_value,
    )
)
moisture_rgb = stretch_composite(
    composite_by_wavelengths(
        img,
        wavelengths,
        (1610, 842, 665),
        invalid_mask=invalid_mask,
        ignore_value=ignore_value,
    )
)
water_rgb = stretch_composite(
    composite_by_wavelengths(
        img,
        wavelengths,
        (705, 560, 490),
        invalid_mask=invalid_mask,
        ignore_value=ignore_value,
    )
)

fig, axes = plt.subplots(2, 2, figsize=(16, 14), subplot_kw={"aspect": "auto"})
fig.suptitle("Airborne composites", fontsize=15)

axes[0, 0].imshow(rgb_default, interpolation="bilinear")
axes[0, 0].set_title("Natural RGB (ENVI default bands)")
axes[0, 0].axis("off")

axes[0, 1].imshow(vegetation_rgb, interpolation="bilinear")
axes[0, 1].set_title("Vegetation composite: 842 / 665 / 560")
axes[0, 1].axis("off")

axes[1, 0].imshow(moisture_rgb, interpolation="bilinear")
axes[1, 0].set_title("Moisture composite: 1610 / 842 / 665")
axes[1, 0].axis("off")

axes[1, 1].imshow(water_rgb, interpolation="bilinear")
axes[1, 1].set_title("Water composite: 705 / 560 / 490")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()


## 3. Airborne Water-Quality and Algae Detection Index Maps

Nine proxy indices are computed from the airborne hyperspectral bands, organised
into two tiers.

**Tier 1 — Cross-sensor indices (airborne + Sentinel-2):**

| Index | Formula | Measures |
|-------|---------|----------|
| **Chl-a red-edge peak** | `R705 - baseline(R665, R740)` | Chlorophyll-a absorption trough near 705 nm |
| **NDCI** | `(R705 - R665) / (R705 + R665)` | Normalised chlorophyll difference |
| **NDRE** | `(R842 - R705) / (R842 + R705)` | Chlorophyll in dense blooms |
| **FAI** | `R842 - baseline(R665, R1610)` | Floating algae / surface scums |
| **NDWI** | `(R560 - R842) / (R560 + R842)` | Water mask (positive = water) |
| **DOC / CDOM proxy** | `R560 / R665` | Dissolved organic carbon |
| **NDTI** | `(R665 - R560) / (R665 + R560)` | Turbidity |

**Tier 2 — Airborne-only indices (narrow-band requirement):**

| Index | Formula | Measures |
|-------|---------|----------|
| **FLH** | `R681 - baseline(R665, R709)` | Chlorophyll fluorescence emission |
| **MCI** | `R709 - baseline(R681, R753)` | Maximum chlorophyll (bloom intensity) |

Each map is displayed with a 2–98 percentile stretch. The histograms and summary
statistics below give an overview of the value distribution across the scene.

In [ ]:
r560 = band_by_wavelength(img, wavelengths, 560, invalid_mask=invalid_mask, ignore_value=ignore_value)
r665 = band_by_wavelength(img, wavelengths, 665, invalid_mask=invalid_mask, ignore_value=ignore_value)
r681 = band_by_wavelength(img, wavelengths, 681, invalid_mask=invalid_mask, ignore_value=ignore_value)
r705 = band_by_wavelength(img, wavelengths, 705, invalid_mask=invalid_mask, ignore_value=ignore_value)
r709 = band_by_wavelength(img, wavelengths, 709, invalid_mask=invalid_mask, ignore_value=ignore_value)
r740 = band_by_wavelength(img, wavelengths, 740, invalid_mask=invalid_mask, ignore_value=ignore_value)
r753 = band_by_wavelength(img, wavelengths, 753, invalid_mask=invalid_mask, ignore_value=ignore_value)
r842 = band_by_wavelength(img, wavelengths, 842, invalid_mask=invalid_mask, ignore_value=ignore_value)
r1610 = band_by_wavelength(img, wavelengths, 1610, invalid_mask=invalid_mask, ignore_value=ignore_value)

index_maps = {
    # cross-sensor indices
    "chl_red_edge_peak": chlorophyll_red_edge_peak(r665, r705, r740),
    "ndci": ndci(r665, r705),
    "ndre": ndre(r705, r842),
    "fai": fai(r665, r842, r1610),
    "ndwi": ndwi(r560, r842),
    "doc_proxy": doc_proxy_green_red(r560, r665),
    "ndti": ndti(r560, r665),
    # airborne-only indices
    "flh": flh(r665, r681, r709),
    "mci": mci(r681, r709, r753),
}

display_cmaps = {
    "chl_red_edge_peak": "viridis",
    "ndci": "viridis",
    "ndre": "YlGn",
    "fai": "RdYlGn",
    "ndwi": "RdBu",
    "doc_proxy": "magma",
    "ndti": "coolwarm",
    "flh": "viridis",
    "mci": "viridis",
}

def finite_limits(array: np.ndarray) -> tuple[float, float]:
    values = np.asarray(array)[np.isfinite(array)]
    if values.size == 0:
        return -1.0, 1.0
    lo, hi = np.percentile(values, [2, 98])
    if lo == hi:
        return float(lo - 1e-6), float(hi + 1e-6)
    return float(lo), float(hi)

n_indices = len(index_maps)
ncols_grid = 3
nrows_grid = (n_indices + ncols_grid - 1) // ncols_grid

fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=(18, 5 * nrows_grid), subplot_kw={"aspect": "auto"})
fig.suptitle("Airborne water-quality and algae detection products", fontsize=15)

for ax, (name, array) in zip(axes.flat, index_maps.items()):
    lo, hi = finite_limits(array)
    image = ax.imshow(array, cmap=display_cmaps[name], vmin=lo, vmax=hi)
    ax.set_title(name)
    ax.axis("off")
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

for ax in axes.flat[n_indices:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

airborne_index_stats = pd.DataFrame(
    [
        {
            "index": name,
            "valid_pixels": int(values.size),
            "mean": float(np.mean(values)) if values.size else np.nan,
            "std": float(np.std(values)) if values.size else np.nan,
            "p05": float(np.percentile(values, 5)) if values.size else np.nan,
            "median": float(np.percentile(values, 50)) if values.size else np.nan,
            "p95": float(np.percentile(values, 95)) if values.size else np.nan,
        }
        for name, array in index_maps.items()
        for values in [array[np.isfinite(array)]]
    ]
)
display(airborne_index_stats)

fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=(18, 4 * nrows_grid))
fig.suptitle("Airborne index histograms", fontsize=15)

for ax, (name, array) in zip(axes.flat, index_maps.items()):
    values = array[np.isfinite(array)]
    ax.hist(values, bins=60, color="steelblue", alpha=0.85)
    ax.set_title(name)
    ax.set_xlabel("Value")
    ax.set_ylabel("Pixel count")
    ax.grid(True, alpha=0.2)

for ax in axes.flat[n_indices:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## 3b. Algae Detection Analysis (Water Pixels Only)

The algae-focused indices are most meaningful over water. This section applies
the **NDWI water mask** (NDWI > 0) to isolate water pixels, then reports
statistics and maps for the five algae-relevant indices:

- **NDWI** separates water from land, creating the mask.
- **NDCI** and **NDRE** respond to dissolved chlorophyll-a in the water column.
- **FAI** specifically detects floating algae mats and surface scums.
- **FLH** (airborne only) directly measures chlorophyll fluorescence emission.
- **MCI** (airborne only) measures the chlorophyll absorption maximum.

A pixel with high NDCI/NDRE, positive FAI, and positive NDWI is a strong algal
bloom candidate. Conversely, high NDTI with low NDCI may indicate sediment
rather than algae.

In [ ]:
# Water mask from NDWI
water_mask = index_maps["ndwi"] > 0.0
water_pixel_count = int(np.sum(water_mask))
total_pixels = int(water_mask.size)
print(f"NDWI water mask: {water_pixel_count:,} water pixels out of {total_pixels:,} total ({100 * water_pixel_count / total_pixels:.1f}%)")

# Algae-focused analysis over water pixels only
algae_indices = ["ndci", "ndre", "fai", "flh", "mci"]
water_algae_stats = pd.DataFrame(
    [
        {
            "index": name,
            "water_pixels": int(np.sum(water_mask & np.isfinite(index_maps[name]))),
            "mean": float(np.nanmean(np.where(water_mask, index_maps[name], np.nan))),
            "std": float(np.nanstd(np.where(water_mask, index_maps[name], np.nan))),
            "p05": float(np.nanpercentile(index_maps[name][water_mask & np.isfinite(index_maps[name])], 5))
            if np.any(water_mask & np.isfinite(index_maps[name]))
            else np.nan,
            "p95": float(np.nanpercentile(index_maps[name][water_mask & np.isfinite(index_maps[name])], 95))
            if np.any(water_mask & np.isfinite(index_maps[name]))
            else np.nan,
        }
        for name in algae_indices
    ]
)
display(water_algae_stats)

# Water-masked algae index maps
algae_cmaps = {"ndci": "viridis", "ndre": "YlGn", "fai": "RdYlGn", "flh": "viridis", "mci": "viridis"}
fig, axes = plt.subplots(2, 3, figsize=(18, 10), subplot_kw={"aspect": "auto"})
fig.suptitle("Algae detection indices — water pixels only", fontsize=15)

for ax, name in zip(axes.flat, algae_indices):
    masked_array = np.where(water_mask, index_maps[name], np.nan)
    lo, hi = finite_limits(masked_array)
    image = ax.imshow(masked_array, cmap=algae_cmaps[name], vmin=lo, vmax=hi)
    ax.set_title(f"{name} (water only)")
    ax.axis("off")
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

# Show the water mask in the last subplot
axes.flat[5].imshow(water_mask, cmap="Blues", interpolation="nearest")
axes.flat[5].set_title("NDWI water mask")
axes.flat[5].axis("off")

plt.tight_layout()
plt.show()

## 4. Sentinel-2 Search and Local Cache

To compare airborne and satellite products, we need a Sentinel-2 scene acquired
as close as possible to the airborne flight date (`2025-06-17`).

**Selection algorithm:**

1. Convert the airborne scene footprint from its native UTM CRS to WGS-84.
2. Query the Microsoft Planetary Computer STAC catalog for Sentinel-2 L2A scenes
   within a ±5-day window around the target date.
3. Filter to scenes with cloud cover below 20%.
4. Score candidates by `|date offset| + 0.01 * cloud_cover`.
5. Select the best-scoring item.

If a live STAC query is not possible (e.g. offline environment), the code falls
back to a previously downloaded local cache under `data/sentinel2/`.

In [ ]:
target_grid = scene_grid_from_envi(meta)
scene_bbox_wgs84 = scene_bbox_wgs84_from_envi(meta)
selected_item_summary = None

try:
    catalog = open_catalog()
    s2_items = search_s2_items(
        scene_bbox_wgs84,
        TARGET_DATE,
        day_window=S2_DAY_WINDOW,
        cloud_limit=S2_CLOUD_LIMIT,
        catalog=catalog,
    )
    s2_candidates = pd.DataFrame([describe_item(item, TARGET_DATE) for item in s2_items])
    s2_item = select_best_item(s2_items, TARGET_DATE)
    selected_item_summary = describe_item(s2_item, TARGET_DATE)
except Exception as exc:
    s2_items = []
    s2_candidates = pd.DataFrame()
    s2_item = None
    print(f"Live Sentinel-2 search unavailable in this environment: {exc}")

cached_band_files = find_cached_band_files(
    SENTINEL2_DIR,
    item_id=s2_item.id if s2_item is not None else None,
)
if not cached_band_files and s2_item is None:
    cached_band_files = find_cached_band_files(SENTINEL2_DIR)

if not s2_candidates.empty:
    display(s2_candidates)
if selected_item_summary is not None:
    display(pd.DataFrame([selected_item_summary]))

if cached_band_files:
    cached_table = pd.DataFrame(
        [
            {"band": band_name, "cached_path": str(path)}
            for band_name, path in cached_band_files.items()
        ]
    )
    display(cached_table)
else:
    print("No complete local Sentinel-2 cache found yet.")


## 5. Sentinel-2 Band Download and Reprojection

Each required Sentinel-2 band (B03, B04, B05, B06, B08, B11) is downloaded as a
COG from Planetary Computer, reprojected to the airborne scene grid (same CRS,
transform, and pixel dimensions), and saved as a local GeoTIFF. Subsequent runs
skip the download if the cached files are already present.

The reprojection step uses bilinear resampling and aligns every Sentinel-2 pixel
exactly to the airborne grid, so that later pixel-by-pixel comparison is valid
without any further resampling.

In [ ]:
if s2_item is not None:
    SENTINEL2_DIR.mkdir(parents=True, exist_ok=True)
    s2_band_files = {
        band_name: download_band(s2_item, band_name, SENTINEL2_DIR, target_grid)
        for band_name in SENTINEL2_REQUIRED_BANDS
    }
else:
    s2_band_files = cached_band_files

if not s2_band_files:
    print("Skipping Sentinel-2 stack load because no live item or cached band set is available.")
    s2_stack = {}
else:
    s2_stack = load_local_stack(s2_band_files)
    download_table = pd.DataFrame(
        [
            {
                "band": band_name,
                "path": str(path),
                "shape": tuple(s2_stack[band_name].shape),
            }
            for band_name, path in s2_band_files.items()
        ]
    )
    display(download_table)


## 6. Cross-Sensor Alignment and Overlap Mask

Both data sources now sit on the same pixel grid. Before comparing them, we build
an **overlap mask** that keeps only pixels valid in both the airborne and
Sentinel-2 products. Pixels that are NaN, zero, or outside the scene extent in
either source are excluded.

The Sentinel-2 index products use the same formulas as the airborne ones, but
with the Sentinel-2 band mapping (B03→560, B04→665, B05→705, B06→740,
B08→842, B11→1610). Only the **7 cross-sensor indices** are compared here;
FLH and MCI are airborne-only and are excluded from the comparison.

In [ ]:
comparison_grid = target_grid

# Only cross-sensor indices go into the comparison (exclude airborne-only FLH, MCI)
CROSS_SENSOR_INDICES = ["chl_red_edge_peak", "ndci", "ndre", "fai", "ndwi", "doc_proxy", "ndti"]

airborne_products = {
    name: align_array_to_grid(index_maps[name], target_grid, comparison_grid)
    for name in CROSS_SENSOR_INDICES
}

if not s2_stack:
    s2_reflectance = {}
    sentinel_products = {}
    overlap_mask = np.zeros((comparison_grid.height, comparison_grid.width), dtype=bool)
    print("Skipping comparison setup because Sentinel-2 data is unavailable.")
else:
    s2_reflectance = {
        band_name: np.where(np.asarray(array, dtype=np.float32) > 0.0, np.asarray(array, dtype=np.float32), np.nan)
        for band_name, array in s2_stack.items()
    }
    sentinel_products = {
        "chl_red_edge_peak": chlorophyll_red_edge_peak(
            s2_reflectance["B04"],
            s2_reflectance["B05"],
            s2_reflectance["B06"],
        ),
        "ndci": ndci(s2_reflectance["B04"], s2_reflectance["B05"]),
        "ndre": ndre(s2_reflectance["B05"], s2_reflectance["B08"]),
        "fai": fai(s2_reflectance["B04"], s2_reflectance["B08"], s2_reflectance["B11"]),
        "ndwi": ndwi(s2_reflectance["B03"], s2_reflectance["B08"]),
        "doc_proxy": doc_proxy_green_red(s2_reflectance["B03"], s2_reflectance["B04"]),
        "ndti": ndti(s2_reflectance["B03"], s2_reflectance["B04"]),
    }
    overlap_mask = build_overlap_mask(airborne_products, sentinel_products)

    shape_table = pd.DataFrame(
        [
            {"source": "airborne", "product": name, "shape": tuple(array.shape)}
            for name, array in airborne_products.items()
        ]
        + [
            {"source": "sentinel2", "product": name, "shape": tuple(array.shape)}
            for name, array in sentinel_products.items()
        ]
    )
    overlap_summary = pd.DataFrame(
        [
            {
                "valid_overlap_pixels": int(overlap_mask.sum()),
                "total_pixels": int(overlap_mask.size),
                "coverage_fraction": float(overlap_mask.mean()),
            }
        ]
    )
    display(shape_table)
    display(overlap_summary)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(overlap_mask, cmap="gray", interpolation="nearest")
    ax.set_title("Airborne and Sentinel-2 overlap mask")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Cross-Sensor Comparison Outputs

This section produces the full set of comparison outputs over the valid overlap:

- **Side-by-side maps** — visually compare spatial patterns per index.
- **Overlaid histograms** — compare value distributions.
- **Scatter plots** — each point is one pixel; the 1:1 line shows perfect agreement.
- **Summary statistics table** — bias, MAE, RMSE, and Pearson correlation per index.

All outputs are saved to `data/outputs/comparison/<scene_id>/` as PNGs and a CSV.

In [ ]:
def sample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 50000, seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    if x.size <= max_points:
        return x, y
    rng = np.random.default_rng(seed)
    sample_idx = rng.choice(x.size, size=max_points, replace=False)
    return x[sample_idx], y[sample_idx]

if not sentinel_products or int(overlap_mask.sum()) == 0:
    comparison_stats = pd.DataFrame()
    print("Skipping comparison outputs because no shared valid overlap is available.")
else:
    output_dir = comparison_output_dir(OUTPUTS_ROOT, SCENE_PATH.stem)
    masked_airborne = {
        name: np.where(overlap_mask, array, np.nan)
        for name, array in airborne_products.items()
    }
    masked_sentinel = {
        name: np.where(overlap_mask, array, np.nan)
        for name, array in sentinel_products.items()
    }

    comparison_stats = pd.DataFrame(
        [
            {"index": name, **summarize_pair(masked_airborne[name], masked_sentinel[name], overlap_mask)}
            for name in masked_airborne
        ]
    )
    comparison_stats.to_csv(output_dir / "comparison_stats.csv", index=False)
    display(comparison_stats)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(overlap_mask, cmap="gray", interpolation="nearest")
    ax.set_title("Airborne and Sentinel-2 overlap mask")
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(output_dir / "overlap_mask.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    n_products = len(masked_airborne)
    fig, axes = plt.subplots(n_products, 2, figsize=(14, 4 * n_products), subplot_kw={"aspect": "auto"})
    if n_products == 1:
        axes = np.asarray([axes])
    for row_idx, name in enumerate(masked_airborne):
        airborne_array = masked_airborne[name]
        sentinel_array = masked_sentinel[name]
        combined = np.concatenate([airborne_array[overlap_mask], sentinel_array[overlap_mask]])
        lo, hi = finite_limits(combined)

        left = axes[row_idx, 0]
        right = axes[row_idx, 1]

        image_left = left.imshow(airborne_array, cmap=display_cmaps.get(name, "viridis"), vmin=lo, vmax=hi)
        left.set_title(f"Airborne {name}")
        left.axis("off")
        plt.colorbar(image_left, ax=left, fraction=0.046, pad=0.04)

        image_right = right.imshow(sentinel_array, cmap=display_cmaps.get(name, "viridis"), vmin=lo, vmax=hi)
        right.set_title(f"Sentinel-2 {name}")
        right.axis("off")
        plt.colorbar(image_right, ax=right, fraction=0.046, pad=0.04)

    plt.tight_layout()
    fig.savefig(output_dir / "comparison_maps.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    hist_ncols = 3
    hist_nrows = (n_products + hist_ncols - 1) // hist_ncols
    fig, axes = plt.subplots(hist_nrows, hist_ncols, figsize=(18, 4 * hist_nrows))
    fig.suptitle("Airborne vs Sentinel-2 histograms", fontsize=15)
    for ax, name in zip(axes.flat, masked_airborne):
        airborne_values = masked_airborne[name][overlap_mask]
        sentinel_values = masked_sentinel[name][overlap_mask]
        ax.hist(airborne_values, bins=60, alpha=0.6, label="Airborne", color="steelblue")
        ax.hist(sentinel_values, bins=60, alpha=0.45, label="Sentinel-2", color="darkorange")
        ax.set_title(name)
        ax.set_xlabel("Value")
        ax.set_ylabel("Pixel count")
        ax.legend()
        ax.grid(True, alpha=0.2)
    for ax in axes.flat[n_products:]:
        ax.set_visible(False)
    plt.tight_layout()
    fig.savefig(output_dir / "comparison_histograms.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    scatter_nrows = (n_products + hist_ncols - 1) // hist_ncols
    fig, axes = plt.subplots(scatter_nrows, hist_ncols, figsize=(18, 5 * scatter_nrows))
    fig.suptitle("Airborne vs Sentinel-2 scatter plots", fontsize=15)
    for ax, name in zip(axes.flat, masked_airborne):
        airborne_values = masked_airborne[name][overlap_mask]
        sentinel_values = masked_sentinel[name][overlap_mask]
        sample_airborne, sample_sentinel = sample_xy(airborne_values, sentinel_values)
        ax.scatter(sample_airborne, sample_sentinel, s=2, alpha=0.15, color="black")
        combined = np.concatenate([sample_airborne, sample_sentinel])
        lo, hi = finite_limits(combined)
        ax.plot([lo, hi], [lo, hi], color="crimson", linewidth=1.5)
        ax.set_title(name)
        ax.set_xlabel("Airborne")
        ax.set_ylabel("Sentinel-2")
        ax.grid(True, alpha=0.2)
    for ax in axes.flat[n_products:]:
        ax.set_visible(False)
    plt.tight_layout()
    fig.savefig(output_dir / "comparison_scatter.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    output_table = pd.DataFrame(
        [{"file": path.name, "path": str(path)} for path in sorted(output_dir.iterdir())]
    )
    display(output_table)
    print(f"Saved comparison outputs to {output_dir}")

## 8. Spectral Library References

The spectral library was built interactively using `viewer.py`: the user selects
rectangular ROIs over representative land-cover patches (water, forest, green
vegetation, etc.), labels them, and exports them. The
`scripts/rebuild_spectral_library.py` script then aggregates per-ROI samples into
per-class mean and standard-deviation spectra.

This section loads those class summaries and plots them. The mean spectra serve
as the reference endmembers for the SAM classification that follows.

> **Note:** If no spectral-library summaries are found, the SAM sections will be
> skipped. Use the viewer to export at least one ROI per class, then run the
> rebuild script before re-executing this notebook.

In [ ]:
reference_library = load_library_summaries(SPECTRAL_LIBRARY_DIR)

if not reference_library:
    reference_table = pd.DataFrame()
    print("No spectral-library summaries found. Export ROI samples from viewer.py and rebuild the library before running the SAM section.")
else:
    reference_table = pd.DataFrame(
        [
            {
                "class_name": class_name,
                "sample_count": reference["sample_count"],
                "wavelength_count": int(reference["wavelengths"].size),
                "summary_path": str(reference["summary_path"]),
            }
            for class_name, reference in reference_library.items()
        ]
    )
    display(reference_table)

    fig, ax = plt.subplots(figsize=(12, 6))
    for class_name, reference in reference_library.items():
        mean_spectrum = np.asarray(reference["mean_spectrum"], dtype=np.float64)
        std_spectrum = np.asarray(reference["std_spectrum"], dtype=np.float64)
        ax.plot(reference["wavelengths"], mean_spectrum, linewidth=1.8, label=class_name)
        if np.any(np.isfinite(std_spectrum)):
            spread = np.nan_to_num(std_spectrum, nan=0.0)
            ax.fill_between(
                reference["wavelengths"],
                mean_spectrum - spread,
                mean_spectrum + spread,
                alpha=0.15,
            )
    ax.set_title("Spectral-library class summaries")
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Reflectance / scaled value")
    ax.grid(True, alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()


## 9. Calibration and Reference Resampling

Airborne and Sentinel-2 reflectance values differ due to differences in spectral
response functions, atmospheric paths, and radiometric calibration. Before
applying SAM to Sentinel-2 data, two preparation steps are needed:

1. **Reference resampling:** The full-resolution airborne reference spectra
   (hundreds of bands) are interpolated to the six Sentinel-2 band wavelengths
   (560, 665, 705, 740, 842, 1610 nm).
2. **Per-band linear calibration:** A gain and offset are fitted per band via
   linear regression over the valid overlap pixels, mapping Sentinel-2
   reflectance to the airborne scale. Outlier trimming (5th–95th percentile)
   improves robustness.

The calibration parameters and resampled reference spectra are saved to
`data/outputs/comparison/<scene_id>/`.

In [ ]:
airborne_s2_band_stack = {
    "B02": band_by_wavelength(img, wavelengths, 490, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B03": band_by_wavelength(img, wavelengths, 560, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B04": band_by_wavelength(img, wavelengths, 665, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B05": band_by_wavelength(img, wavelengths, 705, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B06": band_by_wavelength(img, wavelengths, 740, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B07": band_by_wavelength(img, wavelengths, 783, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B08": band_by_wavelength(img, wavelengths, 842, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B8A": band_by_wavelength(img, wavelengths, 865, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B11": band_by_wavelength(img, wavelengths, 1610, invalid_mask=invalid_mask, ignore_value=ignore_value),
    "B12": band_by_wavelength(img, wavelengths, 2190, invalid_mask=invalid_mask, ignore_value=ignore_value),
}
sam_output_dir = comparison_output_dir(OUTPUTS_ROOT, SCENE_PATH.stem)

if not reference_library or not s2_reflectance or int(overlap_mask.sum()) == 0:
    calibration_params = {}
    calibration_table = pd.DataFrame()
    resampled_reference_table = pd.DataFrame()
    sentinel_calibrated_stack = {}
    airborne_reference_spectra = {}
    sentinel_reference_spectra = {}
    print("Skipping calibration because references or Sentinel-2 overlap are unavailable.")
else:
    airborne_reference_spectra = {}
    sentinel_reference_spectra = {}
    for class_name, reference in reference_library.items():
        airborne_reference = align_reference_to_wavelengths(
            reference["mean_spectrum"],
            reference["wavelengths"],
            wavelengths,
        )
        airborne_reference_spectra[class_name] = airborne_reference
        sentinel_reference_spectra[class_name] = resample_airborne_spectrum_to_s2(
            reference["mean_spectrum"],
            reference["wavelengths"],
        )

    calibration_params = fit_bandwise_linear_calibration(
        airborne_s2_band_stack,
        s2_reflectance,
        overlap_mask,
    )
    calibration_table = pd.DataFrame(
        [
            {"band": band_name, **params}
            for band_name, params in calibration_params.items()
        ]
    )
    resampled_reference_table = pd.DataFrame(
        [
            {
                "class_name": class_name,
                **{
                    band_name: float(sentinel_reference_spectra[class_name][band_idx])
                    for band_idx, band_name in enumerate(SENTINEL2_SAM_BANDS)
                },
            }
            for class_name in sentinel_reference_spectra
        ]
    )
    sentinel_calibrated_stack = apply_bandwise_linear_calibration(s2_reflectance, calibration_params)

    calibration_table.to_csv(sam_output_dir / "calibration_params.csv", index=False)
    resampled_reference_table.to_csv(sam_output_dir / "sam_resampled_references.csv", index=False)
    display(calibration_table)
    display(resampled_reference_table)

## 10. SAM Angle Maps and Classification

Spectral Angle Mapping (SAM) measures the angle between a reference spectrum and
each pixel's spectrum in N-dimensional space. A smaller angle means a closer
spectral match to the reference class. SAM is insensitive to brightness
differences, making it suitable for cross-sensor comparison.

For each class in the spectral library:

- **Airborne SAM** uses the full airborne reference aligned to the scene's
  wavelength grid.
- **Sentinel-2 SAM** uses the resampled reference applied to the calibrated
  Sentinel-2 stack.

The final class map assigns each pixel to the class with the smallest spectral
angle. Per-class angle statistics and pixel counts are saved alongside the maps.

In [ ]:
if not airborne_reference_spectra or not sentinel_calibrated_stack or int(overlap_mask.sum()) == 0:
    sam_angle_stats = pd.DataFrame()
    sam_class_counts = pd.DataFrame()
    print("Skipping SAM outputs because references, calibration, or overlap data are unavailable.")
else:
    airborne_cube = img.open_memmap(interleave="bip")
    airborne_angle_maps = {
        class_name: spectral_angle_image(
            airborne_reference_spectra[class_name],
            airborne_cube,
            ignore_value=ignore_value,
            invalid_mask=invalid_mask,
            pixel_mask=overlap_mask,
            chunk_rows=16,
        )
        for class_name in airborne_reference_spectra
    }
    sentinel_calibrated_array = stack_named_bands(sentinel_calibrated_stack, SENTINEL2_SAM_BANDS)
    sentinel_angle_maps = {
        class_name: spectral_angle_image(
            sentinel_reference_spectra[class_name],
            sentinel_calibrated_array,
            pixel_mask=overlap_mask,
            chunk_rows=256,
        )
        for class_name in sentinel_reference_spectra
    }

    airborne_sam = sam_classification(airborne_angle_maps)
    sentinel_sam = sam_classification(sentinel_angle_maps)
    sam_angle_stats = pd.DataFrame(
        [
            {"class_name": class_name, **summarize_pair(airborne_angle_maps[class_name], sentinel_angle_maps[class_name], overlap_mask)}
            for class_name in airborne_angle_maps
        ]
    )
    sam_class_counts = pd.DataFrame(
        [
            {
                "class_name": class_name,
                "airborne_pixels": int(np.sum(airborne_sam["class_index"] == idx)),
                "sentinel_pixels": int(np.sum(sentinel_sam["class_index"] == idx)),
            }
            for idx, class_name in enumerate(airborne_sam["class_names"], start=1)
        ]
    )

    sam_angle_stats.to_csv(sam_output_dir / "sam_angle_stats.csv", index=False)
    sam_class_counts.to_csv(sam_output_dir / "sam_class_counts.csv", index=False)
    display(sam_angle_stats)
    display(sam_class_counts)

    fig, axes = plt.subplots(len(airborne_angle_maps), 2, figsize=(14, 4 * len(airborne_angle_maps)), subplot_kw={"aspect": "auto"})
    if len(airborne_angle_maps) == 1:
        axes = np.asarray([axes])
    for row_idx, class_name in enumerate(airborne_angle_maps):
        airborne_angle = np.where(overlap_mask, airborne_angle_maps[class_name], np.nan)
        sentinel_angle = np.where(overlap_mask, sentinel_angle_maps[class_name], np.nan)
        combined = np.concatenate([airborne_angle[overlap_mask], sentinel_angle[overlap_mask]])
        lo, hi = finite_limits(combined)

        left = axes[row_idx, 0]
        right = axes[row_idx, 1]

        image_left = left.imshow(airborne_angle, cmap="viridis_r", vmin=lo, vmax=hi)
        left.set_title(f"Airborne SAM angle: {class_name}")
        left.axis("off")
        plt.colorbar(image_left, ax=left, fraction=0.046, pad=0.04)

        image_right = right.imshow(sentinel_angle, cmap="viridis_r", vmin=lo, vmax=hi)
        right.set_title(f"Sentinel-2 calibrated SAM angle: {class_name}")
        right.axis("off")
        plt.colorbar(image_right, ax=right, fraction=0.046, pad=0.04)

    plt.tight_layout()
    fig.savefig(sam_output_dir / "sam_angle_maps.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    air_classes = np.where(overlap_mask, airborne_sam["class_index"], 0)
    s2_classes = np.where(overlap_mask, sentinel_sam["class_index"], 0)
    n_classes = len(airborne_sam["class_names"])
    class_cmap = plt.cm.get_cmap("tab10", n_classes + 1)
    class_vmin = -0.5
    class_vmax = n_classes + 0.5

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 6),
        subplot_kw={"aspect": "auto"},
        gridspec_kw={"wspace": 0.08},
    )
    image_left = axes[0].imshow(air_classes, cmap=class_cmap, vmin=class_vmin, vmax=class_vmax)
    axes[0].set_title("Airborne SAM classes")
    axes[0].axis("off")

    image_right = axes[1].imshow(s2_classes, cmap=class_cmap, vmin=class_vmin, vmax=class_vmax)
    axes[1].set_title("Sentinel-2 calibrated SAM classes")
    axes[1].axis("off")

    fig.subplots_adjust(right=0.86)
    cax = fig.add_axes([0.88, 0.18, 0.02, 0.64])
    cbar = fig.colorbar(image_right, cax=cax, ticks=np.arange(n_classes + 1))
    cbar.ax.set_yticklabels(["none"] + airborne_sam["class_names"])
    cbar.ax.tick_params(labelsize=11)
    fig.savefig(sam_output_dir / "sam_classification.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    sam_output_table = pd.DataFrame(
        [
            {"file": path.name, "path": str(path)}
            for path in sorted(sam_output_dir.iterdir())
            if path.name.startswith("sam_") or path.name == "calibration_params.csv"
        ]
    )
    display(sam_output_table)
    print(f"Saved SAM outputs to {sam_output_dir}")


## Summary

This notebook executed the full Lab 5 pipeline:

1. **Airborne scene** — loaded, inspected, and visualised as composites.
2. **Water-quality indices** — Chl-a red-edge peak, NDCI, DOC proxy, and NDTI
   computed for the airborne data.
3. **Algae detection indices** — NDRE, FAI, NDWI, FLH, and MCI added for bloom
   detection. NDWI provides the water mask; FLH and MCI are airborne-only
   diagnostics.
4. **Algae analysis** — NDWI water mask applied; algae indices analysed over
   water pixels only.
5. **Sentinel-2 acquisition** — closest scene found via STAC, downloaded, and
   reprojected to the airborne grid.
6. **Cross-sensor comparison** — side-by-side maps, histograms, scatter plots,
   and summary statistics produced for 7 cross-sensor indices over the valid
   overlap.
7. **SAM classification** — spectral-library references used to classify both
   airborne and calibrated Sentinel-2 data.

All outputs are saved under `data/outputs/comparison/<scene_id>/` for further
analysis or reporting.